### Phase 2 (Data Understanding)

### Käsikirjoitus datan alustavaan käsittelyyn ja hahmottamiseen

1. Datan läpikäynti lataussivustolta saatavan informaation perusteella excelissä ja sen perusteella ominaisuuksien alustava valinta
   - [sarakkeet excelöitynä](../Tyosto/Featuret.xlsx)
   - [sarakkeet csv:nä](../Tyosto/CSV_Featuret.csv)

2. Mukaan otettavat ominaisuudet valittiin seuraavin perustein
   - poistettiin ominaisuudet joissa merkittävä määrä puuttuvia arvoja > 16%
   - ryhmän perstuntuma mitkä ovat myyntihintaan ja aikaan vaikuttavia tekijöitä
   - (ensimmäisessä vaiheessa) mukaan kelpuutettiin 19 ominaisuutta

3. Datatiedoston koon pienentäminen, mahdollistaa sujuvan käsittelyn kotikoneella
   - Otetaan vain tehtävään valitut sarakkeet mukaan (kts. excel)
   - Sarakkeiden datatyypit vähemmän muistia varaaviksi
   - Luetaan muistiin palasissa ja tallennetaan yhdeksi parquet tiedostoksi

4. Luetaan parquet tiedosto uudestaan dataframeksi, työstetään:
   - duplikaattien tunnistus ja poisto (yli 113 k)
   - perustiedot datasta, etenkin sarakekohtaiset puuttuvat arvot ja uniikit olennaisia
   - tarkastellaan lähemmin str tietotyypin sarakkeiden uniikkeja arvoja (ei major_options)
      - kirjoitusvirheiden tai ylimääräisten välilyöntien havaitseminen
      - etsitään mahdollisuuksia klusteroida/niputtaa sarakkeen arvoja jotta saadaan uniikkien määrää pienemmäksi
      - havaitaan tyhjät arvot per sarake, käsittelyn suunnittelu
         - ryhmittely auttaa
      - ajatusta outliereistä, jotain arvoja vain muutama - vääristääkö mallia

   - haetaan raja-arvoja numeraalisista sarakkeista price, year, mileage, engine_displacement, horsepower. **lon ja lat** jätetään tässä huomioimatta, ne käydään läpi yhdistämällä ne toiseen aineistoon
      - outlierien tunnistus
 
   - visualisoinnit tässä vaiheessa
      - outlierit
      - minkä vuosimallin autoja
         
   - tilastolliset yhteenvedot 
   - ryhmittelyt, mahdollistaa puuttuvien arvojen tunnistamisen ja imputoinnin hahmottelun
      
  
6. Tehdään analyysi datan ominaisuuksista ja pyritään tunnistamaan prosessoinille olennaisimmat tehtävät

   - tyhjien arvojen käsittely
   - poikkeavien arvojen käsittely
   - uusien ominaisuuksien luominen 

    

### Pienennetään datatiedoston kokoa 10 GB -> 90 MB
    - dtypes
    - vain valitut sarakkeet
    - vain osa kerralla muistiin (chunk) ja tallennetaan parquet tiedostoksi


In [ ]:
#import pandas as pd


#df_test = pd.read_csv("../../../archive/used_cars_data.csv", header=0, nrows=1000000 )



In [ ]:


import pandas as pd
import glob
import os
import shutil

# numeeristen sarakkeiden dtypet
dtypes = {
    "daysonmarket": "int16",
    "engine_displacement": "float32", #pandasissa ei voi käyttää float16
    "horsepower": "float32",
    "latitude": "float32",
    "longitude": "float32",
    "mileage": "float32",
    "price": "float32",
    "year": "int16"
}

# valittavat sarakkeet
columns = [
    'body_type', 'daysonmarket','engine_displacement', 'engine_type','mileage',
    'fuel_type','horsepower','latitude','longitude','major_options',
    'make_name','model_name','price','transmission_display','wheel_system',
    'year', 'vin', 'listing_id', 'listed_date'
]

chunksize = 200_000

# --------------------------------------
# CLEAN OUTPUT FOLDER
# --------------------------------------
if os.path.exists("./processed"):
    shutil.rmtree("./processed")

os.makedirs("./processed", exist_ok=True)

# --------------------------------------
# READ → CAST → SAVE PARQUET
# --------------------------------------
i = 0

for chunk in pd.read_csv(
        "../../../archive/used_cars_data.csv",
        chunksize=chunksize,
        usecols=columns,
        dtype=dtypes
    ):

    # string-sarakkeet tehokkaammiksi
    object_cols = chunk.select_dtypes(include=["object","string"]).columns
    chunk[object_cols] = chunk[object_cols].astype("string")

    chunk.to_parquet(f"../../../processed/chunk_{i}.parquet", index=False)

    print(f"Tallennettu: chunk_{i}.parquet")
    i += 1


# --------------------------------------
# LOAD FINAL DATAFRAME
# --------------------------------------
files = sorted(glob.glob("../../../processed/chunk_*.parquet"))

df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

#Tallennetaan levylle niin ei tartte joka kerta ajaa
df.to_parquet("../../../archive/df.parquet", index=False)

print("Lopullisen DataFramen muoto:", df.shape)

### Duplikaattiarvojen etsiminen ja poistaminen
- sarakkeet "vin" ja "listing_id" pitäisi olla uniikkeja, mutta molemmat tai jompi kumpi niistä olivat samoja 80 rivin kohdalla - selkeät duplikaatit 
    - vin on autokohtainen, eli periatteessa samaa autoa voitasiin myydä useaan kertaan, mutta 65 muuta identtistä saraketta tekevät sen että kyseessä on duplikaatti vaikka samaa autoa oltaisiinkin myymässä. Esim. kilometrit, listauspäivä.
- melko varmasti duplikaatteja ovat ne joissa loput 65 saraketta ovat identtisiä, niitä löytyi yli 113 k. Ne poistettiin ennen datan pyörittelyä.

In [2]:
import pandas as pd
df = pd.read_parquet("../../../archive/df.parquet")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000040 entries, 0 to 3000039
Data columns (total 19 columns):
 #   Column                Dtype  
---  ------                -----  
 0   vin                   string 
 1   body_type             string 
 2   daysonmarket          int16  
 3   engine_displacement   float32
 4   engine_type           string 
 5   fuel_type             string 
 6   horsepower            float32
 7   latitude              float32
 8   listed_date           string 
 9   listing_id            int64  
 10  longitude             float32
 11  major_options         string 
 12  make_name             string 
 13  mileage               float32
 14  model_name            string 
 15  price                 float32
 16  transmission_display  string 
 17  wheel_system          string 
 18  year                  int16  
dtypes: float32(6), int16(2), int64(1), string(10)
memory usage: 857.9 MB


In [3]:
#Etsitään ja poistetaan duplikaatit, ja poistetaan sarakkeet 'vin' ja 'listing_id' -> varmistettiin että duplikaatit johtuivat ko. sarakkeista

import pandas as pd #tuodaan tämä tässäkin, ei tarvitse ajaa ekaa notea tämän takia
df = pd.read_parquet("../../../archive/df.parquet")

print("Rivimäärä ennen duplikaattien poistoa:", df.shape[0])

print("Duplikaatit jos vin ja listing_id mukana :", df[df.duplicated(keep=False)].shape[0])

df_cop = df.copy()


cols_for_duplicates = df_cop.columns.difference(["vin", "listing_id"])

dupes = df_cop[df_cop.duplicated(subset=cols_for_duplicates, keep=False)]


print("Duplikaattien määrä: ", dupes.shape[0] / 2)  # montako identtistä paria

# 2. Poista ei-sortattavat sarakkeet, kuten listamaiset major_options
#safe_cols = [col for col in df_test.columns if df_test[col].dtype != "object" or col != "major_options" or col != "vin" or col != "listing_id"]

# 3. Järjestä duplikaatit safe_cols-sarakkeilla
#dupes_sorted = dupes.sort_values(by=safe_cols)

#df_test = df_test.drop_duplicates(ignore_index=True) #Säilytetään duplikaateista ensimmäinen esiintymä

df_unik = df_cop.drop_duplicates(subset=cols_for_duplicates, ignore_index=True)

print("\nRivimäärä duplikaattien poiston jälkeen:", df_unik.shape[0])

df = df_unik.drop(columns=["vin", "listing_id"])


df.shape


df.info()


Rivimäärä ennen duplikaattien poistoa: 3000040
Duplikaatit jos vin ja listing_id mukana : 80
Duplikaattien määrä:  113382.0

Rivimäärä duplikaattien poiston jälkeen: 2862233
<class 'pandas.DataFrame'>
RangeIndex: 2862233 entries, 0 to 2862232
Data columns (total 17 columns):
 #   Column                Dtype  
---  ------                -----  
 0   body_type             string 
 1   daysonmarket          int16  
 2   engine_displacement   float32
 3   engine_type           string 
 4   fuel_type             string 
 5   horsepower            float32
 6   latitude              float32
 7   listed_date           string 
 8   longitude             float32
 9   major_options         string 
 10  make_name             string 
 11  mileage               float32
 12  model_name            string 
 13  price                 float32
 14  transmission_display  string 
 15  wheel_system          string 
 16  year                  int16  
dtypes: float32(6), int16(2), string(9)
memory usage: 731.9

In [4]:
print(df[df["price"] > 250000].shape[0])


(
    df.loc[df["price"] > 150000, ["make_name", "model_name"]]
      .drop_duplicates()
      .to_csv("expensive_unique_models.csv", index=False)
)

print(df[["listed_date"]].dtypes)


1069
listed_date    string
dtype: object


In [ ]:
import pandas as pd

# 1. Tarkista formaatti
invalid_mask = pd.to_datetime(
    df['listed_date'],
    format="%Y-%m-%d",
    errors='coerce'
).isna()

if invalid_mask.any():
    print("Virheellinen päivämääräformaatti seuraavissa riveissä listed_date sarakkeessa:")
    print(df.loc[invalid_mask, 'listed_date'])
else:
    #print("Kaikki arvot listed_date sarakkeessa ovat muodossa YYYY-MM-DD.")

    # 2. Muunna datetimeksi
    df["listed_date"] = pd.to_datetime(
        df["listed_date"],
        format="%Y-%m-%d"
    )

    # 3. Snapshot-päivä
    snapshot_date = pd.to_datetime("2020-09-10")

    # 4. Laske maksimi päivät
    max_days = (snapshot_date - df["listed_date"]).dt.days

    df["gap_days"] = max_days - df["daysonmarket"]

    # 5. Suodata autot, jotka on todennäköisesti myyty
    sold_before_snapshot = df[df["daysonmarket"] < max_days]

    #print(sold_before_snapshot.shape)

    
unique_count = df["gap_days"].nunique()
#print("gap_days‑sarakkeessa on", unique_count, "eri arvoa")

#print(df["gap_days"].nunique(dropna=False))

#print(df["gap_days"].unique())

df_2016 = df[df["listed_date"].dt.year == 2016]

sorted(df_2016["gap_days"].unique())
    
df_2016["gap_days"].value_counts().sort_index()    

df_2012_2015 = df[
    (df["listed_date"].dt.year >= 2010) &
    (df["listed_date"].dt.year <= 2013)
]
print(df_2012_2015[["listed_date", "daysonmarket", "gap_days"]])


Kaikki arvot listed_date sarakkeessa ovat muodossa YYYY-MM-DD.
[ 1 -4 -7 -2  0 -1 -6]
        listed_date  daysonmarket  gap_days
124050   2013-11-05          2500         1
124060   2013-11-05          2500         1
152797   2013-07-04          2624         1
153163   2013-07-04          2624         1
222298   2013-04-03          2716         1
237159   2013-05-20          2669         1
244527   2013-05-01          2688         1
280705   2013-11-08          2497         1
385259   2013-10-11          2527        -1
1440672  2012-09-17          2915         0
1635154  2010-11-29          3573         0
1646161  2013-03-21          2730         0
1801579  2013-12-15          2461         0
2100503  2012-08-06          2957         0
2273409  2013-11-07          2499         0
2412413  2012-07-15          2979         0
2482358  2011-07-23          3337         0
2483347  2010-11-03          3599         0
2588872  2013-09-01          2567        -1
2754708  2012-07-19          2976 

In [12]:

count_sep_2020 = df[
    (df["listed_date"] >= "2019-08-01") &
    (df["listed_date"] <= "2020-09-15")
].shape[0]

print(count_sep_2020)


print(f"Prosentuaalinen osuus kaikista: {count_sep_2020/df_unik.shape[0]:.1%}")


2818002
Prosentuaalinen osuus kaikista: 98.5%


In [20]:

start_date = "2018-01-01"
end_date = "2020-09-15"



df_period = df[
    (df["listed_date"] >= start_date) &
    (df["listed_date"] <= end_date)
]

df_period["listed_date"] = pd.to_datetime(df_period["listed_date"])
monthly_counts = (
    df_period["listed_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)




total_count = monthly_counts.sum()


monthly_counts.to_csv(
    "monthly_counts2.csv",
    header=["Autojen_lukumäärä"],
    index_label="Kuukausi"
)


print(monthly_counts)
print("\nYhteensä:", total_count)



listed_date
2018-01        244
2018-02        289
2018-03        434
2018-04        393
2018-05        511
2018-06        632
2018-07        760
2018-08        980
2018-09       1084
2018-10       1349
2018-11       1561
2018-12       1968
2019-01       2155
2019-02       2191
2019-03       3628
2019-04       3611
2019-05       4896
2019-06       5870
2019-07       8826
2019-08      16003
2019-09      21887
2019-10      31430
2019-11      41212
2019-12      52986
2020-01      64271
2020-02      87379
2020-03     106100
2020-04      55503
2020-05      78317
2020-06     209029
2020-07     504812
2020-08    1040731
2020-09     508342
Freq: M, Name: count, dtype: int64

Yhteensä: 2859384


In [ ]:
#Tarkistetaan mahdolliset ylimääräiset välilyönnit ja listed_date kentän oikea formaatti


invalid_mask = pd.to_datetime(df['listed_date'], format="%Y-%m-%d", errors='coerce').isna()

#Päivämääräformaatin tarkitus
if invalid_mask.any():
    print("Virheellinen päivämääräformaatti seuraavissa riveissä listed_date sarakkeessa:")
    print(df.loc[invalid_mask, 'listed_date'])
else:
    print("Kaikki arvot listed_date sarakkeessa ovat muodossa YYYY-MM-DD.")

#Tarkistusfunktio ylimäääräisille välilyönneille ennen ja jälkeen arvojen tarkistuksen

def has_extra_whitespace(x):
    return isinstance(x, str) and (x != x.strip())

mask_before = df.apply(lambda col: col.map(has_extra_whitespace))
print("Arvot joissa on ylimääräisiä välilyöntejä ennen poistoa:\n")
print(mask_before.sum())          # Montako / sarake
#print("\nRivejä joissa välilyöntejä löydetty:\n") #solun alusta tai lopusta
#print(df[mask_before.any(axis=1)].head())


# --- POISTO: strip kaikista string-arvoista ---

#df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
df = df.assign(**{col: df[col].str.strip() for col in df.select_dtypes(include="string")})

# --- TARKISTUS UUDELLEEN POISTON JÄLKEEN ---

mask_after = df.apply(lambda col: col.map(has_extra_whitespace))
print("\nArvot joissa on ylimääräisiä välilyöntejä poiston jälkeen:\n")
print(mask_after.sum())           # Pitäisi olla nollia



### Perustiedot datasta datatyyppien vaihdon, dublikaattien ja ylimääräisten välilyöntien poiston jälkeen 
- tulokset luettava tekstieditorista


In [ ]:



print("\n# SHAPE (rivit, sarakkeet) #")
print(df.shape)

print("\n# INFO #")
print(df.info())

print("\n# DTYPE-TILANNE #")
print(df.dtypes)

print("\n# PUUTTUVAT ARVOT KPL #")
print(df.isna().sum())

print("\n# PUUTTUVAT ARVOT % #")
print(df.isna().sum()/df.shape[0]*100)

print("\n# UNIIKIT ARVOT / SARAKE #")
print(df.nunique())

print("\n# HEAD #")
print(df.head())

#teen tässä uuden df2:n ettei sotke alkuperäistä, voi ajan alun noteja ilman kikkailua
df2 = df.copy() 
#df2["body_type"] = df2["body_type"].fillna("MISSING")

#df2[
#    df2["body_type"].isna() |
#    df2["body_type"].isin(["", "MISSING"])
#].head()

#täytetään tyhjät arvolla 'MISSING', helpompi havaita 
df2 = df2.replace("", "MISSING").fillna("MISSING")

### Perustietojen tulkinta
- Puuttuvia arvoja 9/17 sarakkeessa 2.1-6.4 %, määrällisesti noin 12 500 - 183 000. Merkittävä määrä, mietittävä keinoja tyhjien arvojen käsittelyyn
- Tietotyypeissä ei tunnistettuja ongelmia, nyt käytössä int, float ja string
- Str sarakkeissa (8 kpl) olevia uniikkeja arvoja on paljon, esimerkiksi eri valmistajia 100 kpl, malleja yli 1400 ja vaihteistotyyppejä 44. Käsittelyä täytyy pohtia. Major_options on erikoistapaus, varusteiden listauksessa ei tunnu olevan yhtenäistä tapaa. 



### Tarkastellaan lähemmin str tietotyypin sarakkeiden uniikkeja ja tyhjiä arvoja, etsitään mahdollisuutta yhdistellä arvoja ja mahdollisia erilaisia kirjoitusmuotoja tai -virheitä

In [ ]:

df2 = df2.replace("", "MISSING").fillna("MISSING")

columns = [
    "body_type", "engine_type", "fuel_type", "make_name",
    "model_name", "transmission_display", "wheel_system", "listed_date"
]

rows = []

for col in columns:
    print("\n" + "="*40)
    print(f"# {col} #")
    print(f"Uniikkeja arvoja (myös 'MISSING'): {df2[col].nunique(dropna=False)}\n")

    counts = df2[col].value_counts(dropna=False)
    print(counts)

    # CSV-tallennus
    for value, count in counts.items():
        rows.append({"column": col, "value": value, "count": count})

    #Lisäosa juuri transmission_display-kohdassa
    if col == "transmission_display":
        ev_counts = df2.loc[df["fuel_type"] == "Electric", "transmission_display"] \
                        .value_counts(dropna=False)

        print("\nSähköautojen vaihteet:")
        print(ev_counts)

        # Tallennetaan nämäkin CSV:hen
        for value, count in ev_counts.items():
            rows.append({
                "column": "transmission_display (Electric)",
                "value": value,
                "count": count
            })

#to_CSV kun printtiä ei oikein pysty lukemaan
pd.DataFrame(rows).to_csv("value_counts.csv", index=False)





In [ ]:
# 1. Mallit per valmistaja
models_per_make = (
    df2.groupby("make_name")["model_name"]
      .unique()
      .reset_index(name="models")
)
models_per_make.to_csv("models_per_make.csv", index=False)
print("Tallennettu: models_per_make.csv")

print(models_per_make)

# 2. Valmistaja–malli–korimalli
make_model_counts = (
    df2.groupby(["make_name", "model_name", "body_type"])
      .size()
      .reset_index(name="count")
      .sort_values("count")
)
make_model_counts.to_csv("make_model_counts.csv", index=False)
print("Tallennettu: make_model_counts.csv")

# 3. Harvinaiset mallit (count < 2)
rare_by_make = make_model_counts[make_model_counts["count"] < 2]
rare_by_make.to_csv("rare_by_make.csv", index=False)
print("Tallennettu: rare_by_make.csv")

#4. Valmistaja–malli–vaihteisto
transmission_display_counts = (
    df2.groupby(["make_name", "model_name", "transmission_display"])
      .size()
      .reset_index(name="count")
      .sort_values("count")
)
transmission_display_counts.to_csv("transmission_display_counts.csv", index=False)
print("Tallennettu: transmission_display_counts.csv")

#4. Valmistaja–malli–vaihteisto
transmission_display_counts = (
    df2.groupby(["make_name", "model_name", "transmission_display"])
      .size()
      .reset_index(name="count")
      .sort_values("count")
)
transmission_display_counts.to_csv("transmission_display_counts.csv", index=False)
print("Tallennettu: transmission_display_counts.csv")








### Str sarakkeiden tarkastelun analyysi
- Silmämääräisen excelissä järjestelyn jälkeen näyttää siltä että puuttuvia arvoja pystyy useimmiten johtamaan vastaavilta (esim. sama valmistaja ja malli) riveiltä.
- Uniikkeja arvoja on todella paljon esimerkiksi vaihteistoissa ja moottorityypeissä -> varmastikin viisainta niputtaa esim. manuaali ja automaatti
- Erilaisia automalleja yhdistelmiä on yli 1900 kpl, joista harvinaisuuksia (alle 10 kpl myytyjä) on yli 1200 ja toisaalta yleisimmät 50 mallia edustavat yli puolta koko aineistosta (1522186/2862233=0,53). Harvinaisuudet täytynee huomioida datan prosessoinnissa ettei malli vääristy


### Käydään läpi kuinka paljon puuttuvia arvoja pystyttäisiin päättelemään vastaavista muista riveistä joissa tieto on
- Lähtökohtaisesti jos merkin ja mallin täsmätessä puuttuvan arvon voisi olettaa olevan sama. 
- Samalla nähdään montako riviä joissa em. päättely ei ole mahdollista 
- Tässä vain str kentät

In [ ]:

#Selvitellään kuinka paljon olisi mahdollista päätellä 'luotettavasti' puuttuvia
# 1. Selvitä ryhmät joissa on vähintään yksi tunnettu vaihteisto

print("\n# PUUTTUVAT ARVOT KPL #")
print(df.isna().sum())

# 1. Ryhmät joissa on vähintään yksi tunnettu vaihteisto
groups_with_info = (
    df.groupby(["make_name", "model_name"])["transmission_display"]
      .apply(lambda s: (s.fillna("").str.strip() != "").any())
)

# 2. Maskit
mask_empty = df["transmission_display"].fillna("").str.strip() == ""

mask_group_has_info = df.set_index(["make_name", "model_name"]).index.isin(
    groups_with_info[groups_with_info].index
)

# 3. Lopulliset rivit
df_missing_but_deducible = df[mask_empty & mask_group_has_info]

print("\nVaihteisto pääteltävissä:", df_missing_but_deducible.shape[0])


# Rivit, joille vaihteistoa EI voida päätellä
mask_not_deducible = mask_empty & (~mask_group_has_info)

df_not_deducible = df[mask_not_deducible]

print("\nVaihteistoa EI voida päätellä:", df_not_deducible.shape[0])




In [ ]:
columns_to_check = ["transmission_display", "engine_type", "fuel_type", "wheel_system"]

results = {}


def find_deducible(df, column):

    """
    Palauttaa kaksi dataframea:
    1) rivit, joilla arvo voidaan päätellä ryhmästä
    2) rivit, joilla arvoa ei voi päätellä
    """
    
    # 1. Ryhmät joissa on vähintään yksi tunnettu arvo
    groups_with_info = (
        df.groupby(["make_name", "model_name"])[column]
          .apply(lambda s: (s.fillna("").str.strip() != "").any())
    )

    # 2. Maskit
    mask_empty = df[column].fillna("").str.strip() == ""

    mask_group_has_info = df.set_index(["make_name", "model_name"]).index.isin(
        groups_with_info[groups_with_info].index
    )

    # 3. Pääteltävissä olevat
    df_deducible = df[mask_empty & mask_group_has_info]

    # 4. Ei pääteltävissä olevat
    df_not_deducible = df[mask_empty & ~mask_group_has_info]

    return df_deducible, df_not_deducible




for col in columns_to_check:
    deducible, not_deducible = find_deducible(df, col)
    results[col] = {
        "deducible": deducible,
        "not_deducible": not_deducible
    }
    print(f"\n=== {col} ===")
    print("Pääteltävissä:", deducible.shape[0])
    print("Ei pääteltävissä:", not_deducible.shape[0])



### Johtopäätöksiä
- Jos nähdään luotettavaksi päätellä puuttuvia arvoja vastaavista malleista, niin on tehtävissä. Vain murto-osassa ei voi tällä logiikalla tehdä päätelmää.

### Str sarakkeiden analyysia
- erilaisia automalleja 1429, joista alle 10 kpl myytyjä 433 kpl ja alle 30 kpl myytyjä 613 kpl, mennee kohinaan mallin näkökulmasta ja voi vääristää
- vaihteistotyyppejä 45 kpl, yhdistetään   
- moottorityyppejä 40 kpl, yhdistetään
- valmistajia joilla on vain muutamia myytyjä autoja
- kirjoitusvirheita/ylimääräisiä välilyöntejä ei näkynyt


### Haetaan numeraalisten sarakkeiden arvojen jakaumaa

In [ ]:


num_cols = [
    "daysonmarket",
    "engine_displacement",
    "horsepower",
    "mileage",
    "price",
    "year"
]

summary = pd.DataFrame({
   
    "min": df[num_cols].min(),
    "p2": df[num_cols].quantile(0.02),
    "median": df[num_cols].median(),
    "p98": df[num_cols].quantile(0.98),
    "max": df[num_cols].max()
})

print(summary)

### Kvantiilien avulla saa nopeasti laskettua ala- ja ylärajoja, joilla pystyy arvioimaan mitä kannattaa leikata pois mallista

### Alla kuvaajia. Mielestäni näistä ei oikein saa mitään irti.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Numeeriset sarakkeet:", list(num_cols))

# --- Outlierien tunnistus IQR-menetelmällä ---
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.02)
    Q3 = df[col].quantile(0.98)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    outlier_summary[col] = len(outliers)

print("\nOutlierien määrä per sarake:")
for col, count in outlier_summary.items():
    print(f"{col}: {count}")

# --- Visualisointi: Boxplotit kapea jakauma ---

perus_sarakkeet = [ 'engine_displacement', 'horsepower','year']
plt.figure(figsize=(12, 4 * len(perus_sarakkeet)))

for i, col in enumerate(perus_sarakkeet, 1):
    plt.subplot(len(perus_sarakkeet), 1, i)
    df.boxplot(column=col)
    plt.title(f"Boxplot: {col}")

plt.tight_layout()
plt.show()

# --- Visualisointi: Boxplotit leveät jakaumat ---
cols_to_log = ["mileage", "price", "daysonmarket"]

df_log = df.copy()
df_log[cols_to_log] = df[cols_to_log].apply(lambda x: np.log1p(x))  # log(1+x)


plt.figure(figsize=(8, 10))

for i, col in enumerate(cols_to_log, 1):
    plt.subplot(len(cols_to_log), 1, i)
    df_log.boxplot(column=col)
    plt.title(f"Boxplot (log-skaalattu): {col}")

plt.tight_layout()
plt.show()


# --- Visualisointi: Histogrammit kapea jakauma---

df[perus_sarakkeet].hist(bins=20, figsize=(15,8), density=True)
plt.suptitle("Histogrammit (density=True)", y=1.02)
plt.tight_layout()
plt.show()


# --- Visualisointi: Histogrammit leveä jakauma---

plt.figure(figsize=(15,8))

for i, col in enumerate(cols_to_log, 1):
    plt.subplot(2, 2, i)
    plt.hist(df_log[col], bins=30)
    plt.title(f"{col} (log-skaalattu histogrammi)")
    plt.yscale("log")        # <-- ratkaisee ison y-akselin ongelman
    plt.xlabel("log(1+x) arvo")
    plt.ylabel("Lukumäärä (log-skaalassa)")

plt.tight_layout()
plt.show()


Tarkastetaan ja visualisoidaan minkä vuosimallin autoja datassa on.

In [ ]:
import matplotlib.pyplot as plt

# --- Määritellään ikäryhmät ---
bins = [1915, 1949, 1979, 1999, 2009, 2014, 2018, 2021]
labels = ['1915-1949', '1950-1979', '1980-1999', '2000-2009', '2010-2014', '2015-2018', '2019-2021']

# Luo uusi sarake vuosiryhmille
df['year_group'] = pd.cut(df['year'], bins=bins, labels=labels, right=True)

year_group_counts = df['year_group'].value_counts().sort_index()

# --- Lasketaan prosenttiosuus koko datasetistä per ryhmä ---
year_group_percent = df['year_group'].value_counts(normalize=True).sort_index() * 100
year_group_percent = year_group_percent.round(2)

# Näytä kaikki rivit ilman katkaisua
pd.set_option('display.max_rows', None)
print("Autojen määrä per vuosiryhmä:")
print(year_group_counts)
print("Autojen osuus per vuosiryhmä (%):")
print(year_group_percent)
pd.reset_option('display.max_rows')

# --- Visualisointi ---
plt.figure(figsize=(12,5))
year_group_percent.plot(kind='bar', color='skyblue')
plt.title('Autojen osuus per vuosiryhmä (%)')
plt.ylabel('Prosentti')
plt.xlabel('Vuosiryhmä')
plt.xticks(rotation=45)
plt.show()

In [ ]:
print(df.isnull().sum().sort_values(ascending=False).to_string())

'Mileage' ja 'fuel_type' yllättävän monta null arvoa, samoin 'horsepower'
Jos fuel_type puuttuu niin saisiko sen tiedon pääteltyä ajoneuvon muista tiedoista?

Testataan kuinka monesta prosentista kunkin vuoden autoista Mileage(mittarilukema) puuttuu:

In [ ]:
missing_mileage_percent = df.groupby('year')['mileage'].apply(lambda x: f"{round(x.isnull().mean() * 100, 2)}%")

pd.set_option('display.max_rows', None)
print(missing_mileage_percent)
pd.reset_option('display.max_rows')


Mittarilukema puuttuu erityisesti uusista ja merkittävän vanhoista autoista. Myös horsepower puuttuu useista riveistä, sille voinee tehdä samalla idealla käsittelyä.

Ratkaisu?: 
- Malleja varten luodaan dataan sarake 'mileage_missing', jota se voi hyödyntää ja ymmärtää näissä olevan jotain hieman erilaista.
- Mallissa ei voi olla kumminkaan nan arvoa joten lisätään kaikkiin puuttuviin 'mileage' arvoihin sen vuoden autojen keskiarvo. Voi jossain määrin väärentää mittarilukemaa erityisen hyvin säilytetylle ja vähän ajetulle vanhalle autolle, mutta uskoisin tämän olevan meidän toteutukseen paras ratkaisu. Myös prosentuaalisesti datasetissä on todella vähän vanhoja autoja



In [ ]:
# Tämä datan esikäsittelyyn:
# df['mileage_missing'] = df['mileage'].isnull().astype(int)
# df['mileage'] = df['mileage'].fillna(df['year'].map(median_by_year))

In [ ]:
fuel_percent = (df['fuel_type'].value_counts(normalize=True, dropna=False) * 100).round(2)

fuel_percent.index = fuel_percent.index.fillna('Missing')

print(fuel_percent)

Datassa voisi tuoda yleisimmän polttoainetyypin autoihin merkin, mallin ja vuosimallin perusteella.
Varsinkin kun datasetti on Yhdysvaltojen käyttöautoista ja esim diesel autot ovat varsin harvinaisia (Bensiinikäyttöisten henkilöautojen osuus n. 90%). Uskoisin tämän toimivan hyvin enkä täten usko, että vääriä polttoainetyyppeja tulisi merkittävän paljon.

Esim. Land Rover Discovery 2019 polttoainetyypit (dieseleitä pieni osuus):

In [ ]:
subset = df[(df['make_name'] == 'Land Rover') & 
            (df['model_name'] == 'Discovery') & 
            (df['year'] == 2019)]

fuel_counts = subset['fuel_type'].value_counts()
print(fuel_counts)

Polttoaine tyypin tuominen:
- tämä huomioidaan prosessoinnissa, vaihe 3

- Ennen imputointia
missing_fuel_before = df['fuel_type'].isnull().mean() * 100
print(f"Fuel_type missing before: {missing_fuel_before:.2f}%")

- Lasketaan merkki-malli-vuosimalli yleisin fuel_type
mode_fuel = df.groupby(['make_name', 'model_name', 'year'])['fuel_type'] \
              .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

fuel_fill = df.set_index(['make_name', 'model_name', 'year']).index.to_series().map(mode_fuel).reset_index(drop=True)

- Imputoi puuttuvat
df['fuel_type'] = df['fuel_type'].fillna(fuel_fill)

- jos ei löytynyt aikasemmalla, nyt pelkkä merkki-malli
mode_fuel_mm = df.groupby(['make_name', 'model_name'])['fuel_type'] \
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

df['fuel_type'] = df['fuel_type'].fillna(
    df.set_index(['make_name', 'model_name']).index.to_series().map(mode_fuel_mm).reset_index(drop=True)
)

- pelkän merkin perusteella
mode_fuel_make = df.groupby('make_name')['fuel_type'] \
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)

df['fuel_type'] = df['fuel_type'].fillna(df['make_name'].map(mode_fuel_make))


- poistetaan loput 0.01% joille ei löydy merkinn perusteella:

df = df.dropna(subset=['fuel_type'])

- Jäljelle jääneet
missing_fuel_after = df['fuel_type'].isnull().mean() * 100
print(f"Fuel_type missing after: {missing_fuel_after:.2f}%")

Outlierit hinnan tai mittarilukeman suhteen?

In [ ]:
print("price-sarakkeen tunnuslukuja:\n",df['price'].describe().apply(lambda x: f"{x:.2f}"))
print("\nmileage-sarakkeen tunnuslukuja:\n",df['mileage'].describe().apply(lambda x: f"{x:.2f}"))

Epärealistisia mileage arvoja on olemassa 12 kpl.
Mielestäni nämä rivit voi suosiosta poistaa

*df = df[df['mileage'] <= 600000]*

In [ ]:
# df = df[df['mileage'] <= 600000]
(df['mileage'] > 600000).sum()

**Koordinaattitiedot lon, lat yhdistetään erilliseen 'piirikunta' aineistoon, joka sijoittaa myyntiliikkeeen piirikuntaan, joita on aineistossa noin 3000 kpl. Aineistossa on mukana myös Alaska ja itsehallintoalueet (Puerto Rico, Guam, Amerikan Samoa...)**

### Yhteenveto ja johtopäätökset

Aineisto koostuu yli 3 miljoonan käytetyn auton ilmoituksen tiedoista. Aineisto on kerätty syyskuussa 2020 Yhdysvaltojen alueelta. Oletusarvoisesti aineistossa on myytyjen autojen tiedot, koska muuten ennustettavat hintatieto ja myyntiaika ovat spekulatiivisia.

Aineistossa on duplikaattien poiston jälkeen yli 2,86 M riviä ja sarakkeita 66 kpl, joista 62 on auton ominaisuuksia. Kaksi sarakkeista on myyntiliikkeen sijainnille ja loput kaksi myyntihinnalle ja myyntiajalle. Projektitehtävän kannalta olennaisiksi sarakkeiksi valikoitui yhteensä 18 kpl, joista 13 kpl on auton ominaisuuksia. Muut sarakkeet ovat hinta, myyntiaika, myyntiliikeen sijainti (2 saraketta) ja päivämäärä, jolloin auto tuli myyntiin.

Aineistossa n. 60% ajoneuvoista ovat hyvin uusia (max 2 vuotta vanhoja aineiston päivämäärästä), ja noin 15% on vuosimallin 2010 tai vanhempia autoja. Polttoainetyypiltään n. 87% on bensiinikäyttöisiä. Sijaintitiedot ovat latitude- ja longitude-koordinaatteina, mikä mahdollistaa alueellisten erojen analysoinnin. 

Data on tekniseltä laadultaan hyvä. Tiedostotyyppien kanssa ei ollut ongelmia eikä datasolujen arvoihin tarvinnut juurikaan tehdä korjauksia. Suurimmat haasteet ovat puuttuvat arvot. Useissa keskeisissä muuttujissa oli merkittävä määrä puuttuvia arvoja. Kaikkiaan 10/18 sarakkeessa on puuttuvia arvoja, prosentuaalisesti 2.1-6.4 % kaikista arvoista eli 12 500 - 183 000. Puuttuvien arvojen käsittely on olennainen tehtävä dataprosessointiin. Tässä dokumentissa on alustavia ajatuksia käsittelystä.

Puuttuvien arvojen lisäksi prosessoinnissa täytyy huomioida kuinka ns. harvinaisuudet käsitellään niin etteivät ne ylikorostu mallissa mutta eivät toisaalta jää huomioimatta. Esimerkiski erilaisia automalleja on reilu 1900 kpl, joista alle 10 kpl myyneitä malleja on yli 1200 kappaletta. Toisaalta yleisimmät 50 mallia edustavat yli puolta koko aineistosta (1522186 riviä/2862233 riviä=0,53). 

Monet autojen ominaisuuksia kuvaavat sarakkeet sisältävät huomattavan paljon uniikkeja arvoja, esimerkiksi erilaisia vaihteistoja on 45 kpl ja moottorityyppejä 40 kpl. Tarkoituksenmukainen käsittely vaatii ryhmittelyä pienempään määrään. Mallien tarkkuuden kannalta myös uusien sarakkeiden johtaminen kahdesta tai useammasta sarakkeesta on hyödyllistä. Sijaintitietojen hyödyntäminen vaatii ulkoisen aineiston, jonka avulla voidaan tarkastella alueellisia eroja.  

Aineistossa ei alustavan analyysin mukaan ole merkittävästi poikkeavia arvoja numeraalisissa sarakkeissa. Mahdolliset harvinaisuudet ja yksittäiset poikkevat arvot voidaan käsitellä prosessoinnissa yksinkertaisilla menetelmillä.

